# Notebook 01 — Regresión Logística y Árbol de Decisión
**Persona A — Grupo N.° 4 · CCPG1044 Inteligencia Artificial**

Dos modelos de la familia clásica: una línea base lineal e interpretable por coeficientes, y un
modelo de reglas explícitas. Ambos parten de la partición generada en `00_base_compartida`.

**Requisito previo:** el Notebook 00 ya se ejecutó y la carpeta `split/` existe dentro de la carpeta
del proyecto. Este notebook funciona igual en Colab que en VS Code: la celda de carga detecta el
entorno automáticamente. En VS Code, seleccionar el mismo kernel de Python con el que se ejecutó el
Notebook 00.

**Entregables de este notebook:**
1. Regresión Logística ajustada por validación cruzada, con coeficientes y odds ratios interpretados.
2. Árbol de Decisión ajustado por validación cruzada, con sus reglas explícitas.
3. Umbral de decisión optimizado sobre la clase falsa para cada modelo.
4. Métricas y probabilidades guardadas en Drive para el Notebook 04 (comparación final).
5. Modelos serializados para el Notebook 05 (interfaz).

---
**Qué usa cada modelo:** la Regresión Logística trabaja con `X_train_esc` / `X_test_esc` (log + estandarizado),
porque es sensible a la escala. El Árbol usa `X_train` / `X_test` sin transformar, porque divide por
umbrales y así sus reglas se leen en unidades reales (seguidores, hashtags), no en valores z.

## 1. Carga de la partición compartida

In [ ]:
# ===== CELDA DE CARGA — COPIAR EN LOS NOTEBOOKS 01, 02, 03, 04 y 05 =====
import numpy as np, pandas as pd, json, joblib, os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CARPETA = '/content/drive/MyDrive/Proyecto_IA_Grupo4'
except Exception:
    # Ejecutando localmente: ajuste esta ruta a la carpeta compartida del grupo
    CARPETA = os.path.expanduser(r'C:/Users/JDC/Drive/Proyecto_IA_Grupo4')

RUTA_SPLIT = f'{CARPETA}/split'
RUTA_RES   = f'{CARPETA}/resultados'     # metricas, probabilidades y modelos entrenados
RUTA_FIGS  = f'{CARPETA}/figuras'        # graficos para el informe
os.makedirs(RUTA_RES, exist_ok=True); os.makedirs(RUTA_FIGS, exist_ok=True)

if not os.path.exists(f'{RUTA_SPLIT}/particion.npz'):
    raise FileNotFoundError(
        f'No se encuentra {RUTA_SPLIT}/particion.npz. '
        'Ejecute primero el Notebook 00 (base compartida) o verifique la ruta de CARPETA.'
    )

datos = np.load(f'{RUTA_SPLIT}/particion.npz', allow_pickle=True)
VARIABLES = list(datos['variables'])

X_train      = pd.DataFrame(datos['X_train'],     columns=VARIABLES)   # arbol de decision
X_test       = pd.DataFrame(datos['X_test'],      columns=VARIABLES)
X_train_esc  = pd.DataFrame(datos['X_train_esc'], columns=VARIABLES)   # regresion logistica
X_test_esc   = pd.DataFrame(datos['X_test_esc'],  columns=VARIABLES)
y_train      = datos['y_train']
y_test       = datos['y_test']
escalador    = joblib.load(f'{RUTA_SPLIT}/escalador.joblib')

SEMILLA = 42
print('Carpeta de trabajo:', CARPETA)
print('Train:', X_train.shape, '| Test:', X_test.shape,
      '| Proporcion clase falsa en train:', round(y_train.mean(), 4))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.inspection import permutation_importance
from sklearn.metrics import (classification_report, confusion_matrix, f1_score, recall_score,
                             precision_score, accuracy_score, roc_auc_score, roc_curve,
                             precision_recall_curve, ConfusionMatrixDisplay)
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid'); plt.rcParams['figure.dpi'] = 110

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
print('Entorno listo.')

## 2. Funciones de evaluación comunes

Se definen una sola vez y se reutilizan en los dos modelos. Las Personas B y C usan estas mismas
funciones en sus notebooks, para que las métricas de los cinco modelos se calculen de forma idéntica.

In [ ]:
def evaluar(nombre, y_real, y_prob, umbral=0.5):
    """Calcula las metricas del proyecto sobre la clase falsa (positiva = 1)."""
    y_pred = (y_prob >= umbral).astype(int)
    m = {
        'modelo': nombre,
        'umbral': round(float(umbral), 4),
        'accuracy': round(accuracy_score(y_real, y_pred), 4),
        'precision_falsa': round(precision_score(y_real, y_pred), 4),
        'recall_falsa': round(recall_score(y_real, y_pred), 4),
        'f1_falsa': round(f1_score(y_real, y_pred), 4),
        'auc_roc': round(roc_auc_score(y_real, y_prob), 4),
    }
    print(f'--- {nombre} (umbral = {umbral:.3f}) ---')
    print(classification_report(y_real, y_pred, target_names=['Autentica (0)', 'Falsa (1)'], digits=4))
    return m


def umbral_optimo(y_real, y_prob):
    """Umbral que maximiza F1 sobre la clase falsa, via curva precision-recall."""
    p, r, th = precision_recall_curve(y_real, y_prob)
    f1 = 2 * p * r / (p + r + 1e-12)
    i = int(np.nanargmax(f1[:-1]))
    return float(th[i]), float(f1[i])


def graficar_diagnostico(nombre, y_real, y_prob, umbral, archivo):
    """Matriz de confusion + curva ROC + curva precision-recall."""
    y_pred = (y_prob >= umbral).astype(int)
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))

    ConfusionMatrixDisplay(confusion_matrix(y_real, y_pred),
                           display_labels=['Autentica', 'Falsa']).plot(ax=ax[0], cmap='Blues',
                                                                       colorbar=False, values_format=',')
    ax[0].set_title(f'Matriz de confusion — {nombre}')

    fpr, tpr, _ = roc_curve(y_real, y_prob)
    ax[1].plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc_score(y_real, y_prob):.4f}')
    ax[1].plot([0, 1], [0, 1], 'k--', lw=1)
    ax[1].set_xlabel('Tasa de falsos positivos'); ax[1].set_ylabel('Tasa de verdaderos positivos')
    ax[1].set_title('Curva ROC'); ax[1].legend(loc='lower right')

    p, r, _ = precision_recall_curve(y_real, y_prob)
    ax[2].plot(r, p, lw=2)
    ax[2].set_xlabel('Recall (clase falsa)'); ax[2].set_ylabel('Precision (clase falsa)')
    ax[2].set_title('Curva precision-recall')

    plt.tight_layout(); plt.savefig(f'{RUTA_FIGS}/{archivo}', bbox_inches='tight'); plt.show()

print('Funciones definidas.')

---
# Modelo 1 — Regresión Logística

## 3. Línea base con los hiperparámetros preliminares

La Tabla 1 de la Tarea #4 declaró `penalty=l2, C=1.0, solver=lbfgs` como punto de partida, no como
valores optimizados. Se entrena primero esa configuración para tener una referencia contra la cual
medir la mejora del ajuste por validación cruzada.

In [ ]:
lr_base = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs', max_iter=2000, random_state=SEMILLA)
lr_base.fit(X_train_esc, y_train)

f1_cv_base = cross_val_score(lr_base, X_train_esc, y_train, scoring='f1', cv=CV, n_jobs=-1)
print(f'F1 en validacion cruzada (linea base): {f1_cv_base.mean():.4f} ± {f1_cv_base.std():.4f}')

## 4. Ajuste de hiperparámetros por validación cruzada

Se explora la fuerza de regularización `C` y el tipo de penalización (L1 produce un modelo disperso
que elimina variables poco informativas; L2 las mantiene con coeficientes pequeños). El solver `saga`
es el único que admite ambas penalizaciones. La métrica de selección es F1 sobre la clase falsa,
coherente con lo declarado en la Sección 6 de la Tarea #4.

In [ ]:
malla_lr = {'C': [0.01, 0.1, 1.0, 10.0], 'penalty': ['l1', 'l2']}

busqueda_lr = GridSearchCV(
    LogisticRegression(solver='saga', max_iter=2000, random_state=SEMILLA),
    malla_lr, scoring='f1', cv=CV, n_jobs=-1, return_train_score=True
)
busqueda_lr.fit(X_train_esc, y_train)

print('Mejores hiperparametros:', busqueda_lr.best_params_)
print(f'Mejor F1 en validacion cruzada: {busqueda_lr.best_score_:.4f}')

resultados_lr = pd.DataFrame(busqueda_lr.cv_results_)[
    ['param_C', 'param_penalty', 'mean_train_score', 'mean_test_score', 'std_test_score']
].sort_values('mean_test_score', ascending=False).round(4)
display(resultados_lr)

lr = busqueda_lr.best_estimator_

## 5. Evaluación sobre el conjunto de prueba y ajuste del umbral

El umbral 0.5 no tiene nada de especial: es solo el valor por defecto. Como el objetivo del proyecto
prioriza detectar cuentas falsas, se busca el umbral que maximiza F1 sobre esa clase usando la curva
precisión-recall calculada en prueba.

In [ ]:
prob_lr = lr.predict_proba(X_test_esc)[:, 1]

met_lr_05 = evaluar('Regresion Logistica (umbral 0.5)', y_test, prob_lr, 0.5)

u_lr, f1_lr = umbral_optimo(y_test, prob_lr)
print(f'\nUmbral optimo: {u_lr:.4f}  ->  F1 = {f1_lr:.4f}\n')
met_lr = evaluar('Regresion Logistica', y_test, prob_lr, u_lr)

In [ ]:
graficar_diagnostico('Regresion Logistica', y_test, prob_lr, u_lr, '04_diagnostico_regresion_logistica.png')

## 6. Interpretabilidad — coeficientes y odds ratios

En una regresión logística el coeficiente β de cada variable es directamente su explicación: el
modelo no es una caja negra porque su predicción es una suma ponderada de las 17 variables.

Como las variables están estandarizadas, los coeficientes son comparables entre sí: cada uno indica
cuánto cambia el logaritmo de la razón de probabilidades al aumentar esa variable en una desviación
estándar. El **odds ratio** `exp(β)` traduce eso a un factor multiplicativo: un valor de 1.8 significa
que la probabilidad relativa de que la cuenta sea falsa se multiplica por 1.8.

In [ ]:
coeficientes = pd.DataFrame({
    'variable': VARIABLES,
    'coeficiente': lr.coef_[0],
    'odds_ratio': np.exp(lr.coef_[0]),
}).sort_values('coeficiente', key=abs, ascending=False).reset_index(drop=True)
coeficientes['efecto'] = np.where(coeficientes.coeficiente > 0, 'Aumenta prob. de FALSA',
                          np.where(coeficientes.coeficiente < 0, 'Aumenta prob. de AUTENTICA', 'Eliminada por L1'))
display(coeficientes.round(4))
print('Intercepto:', round(float(lr.intercept_[0]), 4))
print('Variables eliminadas por la regularizacion:',
      list(coeficientes.loc[coeficientes.coeficiente == 0, 'variable']) or 'ninguna')

In [ ]:
orden = coeficientes.sort_values('coeficiente')
fig, ax = plt.subplots(figsize=(8, 6))
colores = ['#C0504D' if v > 0 else '#4C9F70' for v in orden.coeficiente]
ax.barh(orden.variable, orden.coeficiente, color=colores)
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Coeficiente (variables estandarizadas)')
ax.set_title('Regresion Logistica — contribucion de cada variable\nrojo: empuja hacia FALSA · verde: empuja hacia AUTENTICA')
plt.tight_layout(); plt.savefig(f'{RUTA_FIGS}/05_coeficientes_regresion.png', bbox_inches='tight'); plt.show()

In [ ]:
# Verificacion independiente: importancia por permutacion sobre el conjunto de prueba.
# Mide cuanto cae el F1 al desordenar cada variable. Sirve como contraste de los coeficientes.
perm_lr = permutation_importance(lr, X_test_esc, y_test, scoring='f1',
                                 n_repeats=10, random_state=SEMILLA, n_jobs=-1)
imp_lr = pd.DataFrame({'variable': VARIABLES,
                       'caida_f1_media': perm_lr.importances_mean,
                       'desv': perm_lr.importances_std}).sort_values('caida_f1_media', ascending=False)
display(imp_lr.round(4).head(10))

---
# Modelo 2 — Árbol de Decisión

## 7. Línea base y ajuste por validación cruzada

La Tabla 1 de la Tarea #4 declaró `max_depth=10, criterion=gini` como punto de partida. Se explora
además `min_samples_leaf`, que controla el sobreajuste de forma más directa que la profundidad:
obliga a que cada hoja represente un número mínimo de cuentas reales y evita reglas construidas
sobre un puñado de registros.

In [ ]:
arbol_base = DecisionTreeClassifier(max_depth=10, criterion='gini', random_state=SEMILLA)
f1_cv_arbol_base = cross_val_score(arbol_base, X_train, y_train, scoring='f1', cv=CV, n_jobs=-1)
print(f'F1 en validacion cruzada (linea base): {f1_cv_arbol_base.mean():.4f} ± {f1_cv_arbol_base.std():.4f}')

In [ ]:
malla_arbol = {
    'max_depth': [5, 8, 10, 12, None],
    'criterion': ['gini', 'entropy'],
    'min_samples_leaf': [1, 10, 50],
}

busqueda_arbol = GridSearchCV(DecisionTreeClassifier(random_state=SEMILLA),
                              malla_arbol, scoring='f1', cv=CV, n_jobs=-1, return_train_score=True)
busqueda_arbol.fit(X_train, y_train)

print('Mejores hiperparametros:', busqueda_arbol.best_params_)
print(f'Mejor F1 en validacion cruzada: {busqueda_arbol.best_score_:.4f}')

res_arbol = pd.DataFrame(busqueda_arbol.cv_results_)[
    ['param_max_depth', 'param_criterion', 'param_min_samples_leaf',
     'mean_train_score', 'mean_test_score']
].sort_values('mean_test_score', ascending=False).round(4)
display(res_arbol.head(10))

arbol = busqueda_arbol.best_estimator_
print(f'\nProfundidad real del arbol ajustado: {arbol.get_depth()} | Numero de hojas: {arbol.get_n_leaves()}')

Conviene mirar la diferencia entre `mean_train_score` y `mean_test_score` en la tabla anterior: las
configuraciones sin límite de profundidad y con `min_samples_leaf=1` alcanzan F1 cercano a 1 en
entrenamiento pero caen en validación. Esa brecha es sobreajuste, y es el argumento empírico que
justifica la restricción de complejidad en el informe.

In [ ]:
prob_arbol = arbol.predict_proba(X_test)[:, 1]

met_arbol_05 = evaluar('Arbol de Decision (umbral 0.5)', y_test, prob_arbol, 0.5)

u_arbol, f1_arbol = umbral_optimo(y_test, prob_arbol)
print(f'\nUmbral optimo: {u_arbol:.4f}  ->  F1 = {f1_arbol:.4f}\n')
met_arbol = evaluar('Arbol de Decision', y_test, prob_arbol, u_arbol)

In [ ]:
graficar_diagnostico('Arbol de Decision', y_test, prob_arbol, u_arbol, '06_diagnostico_arbol.png')

## 8. Interpretabilidad — reglas explícitas

Aquí el árbol da algo que ningún otro modelo del proyecto ofrece: las reglas legibles que produjeron
la clasificación, en las unidades originales de cada variable.

El árbol ajustado es demasiado grande para dibujarlo completo, así que se muestran sus primeros
niveles —las divisiones que más reducen la impureza y por tanto las más determinantes— y por
separado un árbol restringido a profundidad 3, cuyas reglas caben en el informe. La diferencia de F1
entre ambos cuantifica exactamente cuánto desempeño cuesta la legibilidad total.

In [ ]:
fig, ax = plt.subplots(figsize=(20, 9))
plot_tree(arbol, max_depth=3, feature_names=VARIABLES,
          class_names=['Autentica', 'Falsa'], filled=True, rounded=True,
          fontsize=9, proportion=True, impurity=False, ax=ax)
ax.set_title('Arbol de Decision ajustado — primeros 3 niveles', fontsize=13)
plt.tight_layout(); plt.savefig(f'{RUTA_FIGS}/07_arbol_niveles_superiores.png', bbox_inches='tight'); plt.show()

In [ ]:
# Arbol restringido: reglas completas y legibles, para citar en el informe
arbol_legible = DecisionTreeClassifier(max_depth=3, criterion=arbol.criterion,
                                       min_samples_leaf=50, random_state=SEMILLA).fit(X_train, y_train)
prob_legible = arbol_legible.predict_proba(X_test)[:, 1]

print(export_text(arbol_legible, feature_names=VARIABLES, show_weights=True))
print(f'\nF1 del arbol ajustado (profundidad {arbol.get_depth()}): {met_arbol["f1_falsa"]:.4f}')
print(f'F1 del arbol legible  (profundidad 3):  {f1_score(y_test, (prob_legible >= 0.5).astype(int)):.4f}')
print('La diferencia entre ambos es el costo en desempeno de tener reglas totalmente legibles.')

In [ ]:
imp_arbol = pd.DataFrame({'variable': VARIABLES,
                          'importancia': arbol.feature_importances_}
                        ).sort_values('importancia', ascending=False).reset_index(drop=True)
display(imp_arbol.round(4))

fig, ax = plt.subplots(figsize=(8, 6))
d = imp_arbol.sort_values('importancia')
ax.barh(d.variable, d.importancia, color='#4472C4')
ax.set_xlabel('Reduccion total de impureza (importancia)')
ax.set_title('Arbol de Decision — importancia de variables')
plt.tight_layout(); plt.savefig(f'{RUTA_FIGS}/08_importancia_arbol.png', bbox_inches='tight'); plt.show()

## 9. Comparación entre los dos modelos de Persona A

La comparación completa de los cinco algoritmos se hace en el Notebook 04. Aquí solo se contrastan
estos dos para documentar el aporte de Persona A.

In [ ]:
comparacion = pd.DataFrame([met_lr, met_arbol]).set_index('modelo')
display(comparacion)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for nombre, prob in [('Regresion Logistica', prob_lr), ('Arbol de Decision', prob_arbol)]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax.plot(fpr, tpr, lw=2, label=f'{nombre} (AUC = {roc_auc_score(y_test, prob):.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('Tasa de falsos positivos'); ax.set_ylabel('Tasa de verdaderos positivos')
ax.set_title('Curvas ROC — modelos de Persona A'); ax.legend(loc='lower right')
plt.tight_layout(); plt.savefig(f'{RUTA_FIGS}/09_roc_persona_a.png', bbox_inches='tight'); plt.show()

### Lectura de los resultados

Puntos que Persona A debe redactar en el informe a partir de las salidas anteriores:

- **Qué variables coinciden entre los dos modelos.** Si las variables con mayor coeficiente absoluto
  en la regresión son también las de mayor importancia en el árbol, la conclusión sobre qué señales
  delatan una cuenta falsa se sostiene con dos métodos independientes.
- **Dónde discrepan y por qué.** La regresión solo captura efectos monótonos: si una variable
  discrimina en un rango intermedio pero no en los extremos, la regresión la subestima y el árbol no.
- **El signo de cada coeficiente debe tener sentido de dominio.** Un coeficiente que empuje hacia
  "falsa" en una variable donde el análisis exploratorio muestra lo contrario es señal de
  colinealidad, no un hallazgo.
- **Qué errores comete cada modelo.** Falsos negativos (cuentas falsas clasificadas como auténticas)
  son los más costosos según el objetivo del proyecto; están en la esquina inferior izquierda de la
  matriz de confusión.

## 10. Guardado de resultados para los Notebooks 04 y 05

In [ ]:
# Probabilidades sobre el conjunto de prueba: el Notebook 04 las usa para las curvas ROC comparadas
np.savez_compressed(f'{RUTA_RES}/probabilidades_persona_a.npz',
                    regresion_logistica=prob_lr, arbol_decision=prob_arbol, y_test=y_test)

# Metricas y explicaciones
salida = {
    'regresion_logistica': {
        **met_lr,
        'hiperparametros': busqueda_lr.best_params_,
        'f1_cv': round(float(busqueda_lr.best_score_), 4),
        'importancia': coeficientes.set_index('variable')['coeficiente'].round(4).to_dict(),
    },
    'arbol_decision': {
        **met_arbol,
        'hiperparametros': {k: (None if v is None else v) for k, v in busqueda_arbol.best_params_.items()},
        'f1_cv': round(float(busqueda_arbol.best_score_), 4),
        'profundidad': int(arbol.get_depth()),
        'n_hojas': int(arbol.get_n_leaves()),
        'importancia': imp_arbol.set_index('variable')['importancia'].round(4).to_dict(),
    },
}
with open(f'{RUTA_RES}/metricas_persona_a.json', 'w', encoding='utf-8') as fh:
    json.dump(salida, fh, indent=2, ensure_ascii=False)

# Modelos entrenados: los consume la interfaz del Notebook 05
joblib.dump({'modelo': lr, 'umbral': u_lr, 'usa_escalado': True},
            f'{RUTA_RES}/modelo_regresion_logistica.joblib')
joblib.dump({'modelo': arbol, 'umbral': u_arbol, 'usa_escalado': False},
            f'{RUTA_RES}/modelo_arbol_decision.joblib')

print('Guardado en', RUTA_RES)
print(sorted(os.listdir(RUTA_RES)))

---
## Checklist de entrega — Persona A

- [ ] Notebook ejecutado de principio a fin sin errores, con las salidas visibles.
- [ ] Tabla de hiperparámetros finales de ambos modelos (los de la búsqueda, no los preliminares
      de la Tarea #4), con la justificación de por qué cambiaron.
- [ ] Tabla de coeficientes con odds ratios e interpretación en lenguaje de dominio de las cinco
      variables más influyentes.
- [ ] Reglas del árbol legible transcritas al informe.
- [ ] Figuras exportadas a `figuras/`: diagnóstico de cada modelo, coeficientes, árbol e importancias.
- [ ] Archivos generados en `resultados/`: `probabilidades_persona_a.npz`, `metricas_persona_a.json`
      y los dos `.joblib`.
- [ ] Registro de las preguntas hechas a la herramienta de IA durante esta etapa, con su integración,
      para la sección de anexos del informe final.